## Code to download NMDC sensor data with their API

In [ ]:
import requests, json

BASE = "http://prod1.nmdc.no/UserInterface/metadata-api"

# Step 1: verify facets endpoint works
r = requests.get(f"{BASE}/getFacets")
print(r.status_code)
print(r.text[:500])

In [ ]:
import requests, json, time

BASE = "http://prod1.nmdc.no/UserInterface/metadata-api"

def search(q="*:*", offset=0, rows=100):
    params = {"q": q, "offset": offset, "rows": rows}
    r = requests.get(f"{BASE}/search", params=params)
    r.raise_for_status()
    return r.json()

# Get first page to check structure
first = search(offset=0, rows=1)
print(json.dumps(first, indent=2)[:1000])  # show structure so we know field names

In [ ]:
import requests, json, time

BASE = "http://prod1.nmdc.no/UserInterface/metadata-api"

def search(q="*:*", offset=0, rows=100):
    params = {"q": q, "offset": offset, "rows": rows}
    r = requests.get(f"{BASE}/search", params=params)
    r.raise_for_status()
    return r.json()

# Get total
first = search(offset=0, rows=1)
total = first["numFound"]
print(f"Total datasets: {total}")

# Paginate through all
all_results = []
batch = 100
for offset in range(0, total, batch):
    print(f"Fetching {offset}/{total}...")
    data = search(offset=offset, rows=batch)
    all_results.extend(data["results"])
    time.sleep(0.3)  # be polite

print(f"Done: {len(all_results)} records")

# Save
with open("nmdc_all.json", "w") as f:
    json.dump(all_results, f, indent=2)

In [ ]:
first = print(all_results[0])  # show first record to check structure
print(json.dumps(first, indent=2))

Fetching all download links

In [ ]:
links = []
for r in all_results:
    urls = r.get("Data_URL", "")
    if urls:
        # URLs are stored as "[url1, url2]" strings
        clean = urls.strip("[]").split(", ")
        for url in clean:
            url = url.strip()
            if url:
                links.append({"id": r["Entry_ID"], "url": url, "doi": r.get("Dataset_DOI", "")})

print(f"Found {len(links)} download links")

# Bulk download
import os
os.makedirs("nmdc_data", exist_ok=True)

for link in links:
    url = link["url"]
    filename = url.split("/")[-1] or link["id"]
    path = f"nmdc_data/{filename}"
    if os.path.exists(path):
        continue
    try:
        r = requests.get(url, stream=True, timeout=60)
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(8192):
                f.write(chunk)
        print(f"✓ {filename}")
    except Exception as e:
        print(f"✗ {link['id']}: {e}")
    time.sleep(0.2)